# Januário Leads — Importação programada via Kaggle (Somente HOMOLOGAÇÃO)

> **Alternativa ao GitHub Actions bloqueado:** notebook que pode ser agendado para executar **diariamente** no Kaggle, sem precisar abrir o Colab todos os dias.

## Segurança antes de começar

- **Supabase exclusivo**: projeto de homologação `prlrxjothrslhswpkoza`. **Nunca** use a chave da produção.
- A chave `sb_secret_...` (ou `service_role` legada) deve ficar guardada em **Kaggle Add-ons > Secrets**, com o nome `OVERTURE_SUPABASE_SERVICE_ROLE_KEY`, e ter acesso autorizado para **este notebook**. Jamais cole a chave em qualquer célula.
- Deixe o notebook **Private**, com **Internet ON**, **Accelerator: None (CPU)**.
- O importador verifica o Supabase antes de escrever, ignora cidades já importadas, registra falhas para tentar novamente depois e interrompe novos municípios quando o armazenamento atinge **88 MiB**. Não promete cobertura completa por localidade.

## Procedimento

1. Importe o arquivo `.ipynb` do repositório no [Kaggle Code](https://www.kaggle.com/code) criando um novo notebook (File > Import Notebook, se disponível). Escolha Python e `Internet: On`.
2. Crie o segredo do projeto STAGING em **Add-ons > Secrets**. Conceda permissão de leitura deste segredo ao notebook.
3. Mantenha `MAX_MUNICIPIOS = 2` e `MAX_MINUTOS = 30` na Célula 1; execute todas as células como teste. Confira `CONFIRMADO NO SUPABASE` e o total no painel da homologação.
4. Após confirmar o teste, altere a Célula 1 para `MAX_MUNICIPIOS = 200` e `MAX_MINUTOS = 480` (8 horas), salve uma nova versão.
5. Em **Settings > Schedule a notebook to run**, selecione **Daily** e salve. O Kaggle executará a cada dia segundo suas regras de uso. Os registros ficam persistidos no Supabase, não na sessão do Kaggle.

## Restrições reais

- O Kaggle permite agendamento **diário**, não a execução 24 horas sem interrupção. Os limites e horários podem mudar.
- Cada execução tenta até 200 municípios **dentro do tempo disponível**; não há garantia de processar todos os 200 numa sessão.
- Falhas de rede, limites de computação, limites do Supabase e ausência de empresas podem suspender importações. Não é possível completar 5.000+ municípios sem estratégia de armazenamento e tempo suficiente.
- O notebook não funciona sem a criação de uma conta Kaggle, autorização do segredo e permissão de internet.
- **Não ative simultaneamente o agendamento no GitHub ou Colab**: serviços diferentes não compartilham controle de concorrência. Manter uma única rotina ativa evita disputa e duplicidade.

## Status

Código criado e verificado no repositório. **A execução agendada no Kaggle não está ativada por esta publicação**. Ela depende da configuração manual e do teste inicial na conta do usuário.


In [ ]:
# CONFIGURAÇÕES: comece testando com 2 municípios.
# Após confirmar o teste, altere para 200 e ative o agendamento diário.
MAX_MUNICIPIOS = 2     # Teste: 2 | Agendado: 200
MAX_MINUTOS = 30       # Teste: 30 | Agendado: 480 (8 horas)
MAX_EMPRESAS = 60      # Amostra máxima por município
assert 1 <= MAX_MUNICIPIOS <= 300
assert 5 <= MAX_MINUTOS <= 600
assert 1 <= MAX_EMPRESAS <= 100
print(f"Pronto: até {MAX_MUNICIPIOS} municípios, {MAX_MINUTOS} minutos, {MAX_EMPRESAS} empresas por cidade.")


In [ ]:
# Internet deve estar ATIVADA em Settings > Session options.
# Sem GPU: CPU basta. Apenas bibliotecas necessárias ao importador.
import sys, subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
    "overturemaps", "geopandas", "shapely", "pyogrio"
], check=True, timeout=900)
print("Dependências instaladas. Nenhum dado foi importado nesta etapa.")


In [ ]:
# Códigos auditáveis do repositório Januário Leads, incorporados sem credenciais.
from pathlib import Path
_IMPORTADOR = (
    "#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipa"
    "lities.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https:/"
    "/<project>.supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/i"
    "mport_overture_br.py --states MG,SP --database-max-mb 400\n\nThe runner covers all 27 federal units by defa"
    "ult. Each state is streamed\nseparately; it does not attempt to download and retain a global Overture file"
    ".\nIt stops BEFORE reaching a configured database threshold, and reports partial\ncoverage honestly. Never "
    "run from a browser or publish the service-role key.\n\"\"\"\nimport argparse\nimport json\nimport os\nfrom pathli"
    "b import Path\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom "
    "urllib import request, error\nfrom urllib.parse import urlparse\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ib"
    "ge.gov.br/organizacao_do_territorio/\"\n    \"malhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef s"
    "tate_boundaries_url(state):\n    \"\"\"IBGE offers one ZIP per UF; do not download the 199 MB national file.\""
    "\"\"\n    if state not in STATE_CODES:\n        raise ValueError(\"Invalid Brazilian state code\")\n    return f"
    "\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\ndef ensure_state_boundaries(state, source=Non"
    "e):\n    if source:\n        path = Path(source)\n        if not path.is_file():\n            raise RuntimeEr"
    "ror(f\"IBGE boundaries file not found: {path}\")\n        return path\n    path = Path(\"data/ibge/\") / f\"{sta"
    "te}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkdir(parents=True, exist_ok=True"
    ")\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloading IBGE municipal boundari"
    "es: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n            request.urlr"
    "etrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise RuntimeError(\"IBGE bou"
    "ndaries download appears empty\")\n            temp.rename(path)\n        finally:\n            temp.unlink(m"
    "issing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE P"
    "I \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\n\n\ndef normalize(value):\n    \"\"\"Match the "
    "TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in"
    " unicodedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    retu"
    "rn \" \".join(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in"
    " (values or []) if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or lin"
    "k shortener is NOT a verified\n# business website. This preserves truthful \"sem site\" prospecting filters."
    "\nNON_WEBSITE_HOSTS = frozenset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"f"
    "acebook.com\", \"fb.me\", \"m.me\", \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube"
    ".com\", \"youtu.be\",\n    \"tiktok.com\", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai"
    "\",\n))\n\n\ndef is_business_website(value):\n    if not isinstance(value, str):\n        return False\n    try:\n"
    "        url = urlparse(value.strip())\n        host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n  "
    "      return (url.scheme in (\"http\", \"https\")\n                and bool(host) and \".\" in host\n            "
    "    and host not in NON_WEBSITE_HOSTS\n                and not any(host.endswith(\".\" + blocked)\n          "
    "                  for blocked in NON_WEBSITE_HOSTS))\n    except ValueError:\n        return False\n\n\ndef fi"
    "rst_website(values):\n    return first([value for value in (values or []) if is_business_website(value)])\n"
    "\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for contactable companies without identif"
    "ied websites.\n\n    Fill unused slots with the best available business contacts. Never promise\n    that a "
    "missing website is proof a business actually has no website.\n    \"\"\"\n    ranked = sorted(records, key=lea"
    "d_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n               and (r.g"
    "et(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site), (maximum * 2"
    " + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    for row in"
    " ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n     "
    "       chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path"
    ", method=\"GET\", payload=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is "
    "not None else None\n    # Legacy service_role JWTs use Authorization; the newer sb_secret_ keys\n    # are "
    "opaque and must be sent only as apikey, never as Bearer JWT.\n    if not (key.startswith(\"eyJ\") or key.sta"
    "rtswith(\"sb_secret_\")):\n        raise RuntimeError(\"Use somente a chave secreta de STAGING (sb_secret_ ou"
    " service_role legada).\")\n    headers = {\n        \"apikey\": key,\n        \"Content-Type\": \"application/json"
    "\",\n        \"Prefer\": \"resolution=merge-duplicates,return=minimal\",\n    }\n    if key.startswith(\"eyJ\"):\n  "
    "      headers[\"Authorization\"] = f\"Bearer {key}\"\n    for attempt in range(5):\n        try:\n            re"
    "q = request.Request(base + path, method=method, headers=headers, data=data)\n            with request.urlo"
    "pen(req, timeout=45) as resp:\n                body = resp.read()\n                return json.loads(body) "
    "if body else None\n        except error.HTTPError as exc:\n            detail = exc.read(300).decode(\"utf-8"
    "\", errors=\"replace\")\n            if exc.code not in (429, 500, 502, 503, 504) or attempt == 4:\n          "
    "      raise RuntimeError(f\"Supabase API {exc.code}: {detail}\") from exc\n        except (error.URLError, T"
    "imeoutError):\n            if attempt == 4:\n                raise\n        time.sleep(min(2 ** attempt, 15)"
    ")\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef database_bytes(base, key):\n    result = api"
    "(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payload={})\n    if not isinstance(resul"
    "t, int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL migration first.\")\n    return res"
    "ult\n\n\ndef status(base, key, state, phase, count=0, message=None):\n    api(base, key, \"/rest/v1/overture_i"
    "mport_progress?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase,\n        \"imported_"
    "rows\": count, \"last_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtim"
    "e()),\n    }])\n\n\ndef municipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = gpd.read_f"
    "ile(\"zip://\" + str(boundaries.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The IBGE "
    "municipal boundary file has no CRS.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    colu"
    "mns = set(frame.columns)\n    if not {\"NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError("
    "f\"Unexpected IBGE municipalities columns: {sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(ST"
    "ATE_CODES)].copy().reset_index(drop=True)\n    return frame\n\n\ndef place_record(feature, municipalities, st"
    "ate):\n    from shapely.geometry import Point\n\n    props = feature.get(\"properties\") or {}\n    coords = (f"
    "eature.get(\"geometry\") or {}).get(\"coordinates\") or []\n    if len(coords) != 2:\n        return None\n    l"
    "on, lat = coords\n    if not isinstance(lon, (int, float)) or not isinstance(lat, (int, float)):\n        r"
    "eturn None\n    name = (props.get(\"names\") or {}).get(\"primary\")\n    # GeoJSON exporters may place the Ove"
    "rture GERS id in Feature.id rather\n    # than properties.id. Preserve either representation.\n    identifi"
    "er = props.get(\"id\") or feature.get(\"id\")\n    if not isinstance(name, str) or not name.strip() or not isi"
    "nstance(identifier, str):\n        return None\n    if props.get(\"operating_status\") == \"permanently_closed"
    "\":\n        return None\n\n    # State bounding boxes overlap: map point to the true municipality.\n    point"
    " = Point(lon, lat)\n    matches = municipalities.sindex.query(point, predicate=\"intersects\")\n    if len(ma"
    "tches) == 0:\n        return None\n    mun = municipalities.iloc[int(matches[0])]\n    if mun[\"SIGLA_UF\"] !="
    " state:\n        return None\n    address = (props.get(\"addresses\") or [{}])[0]\n    if not isinstance(addre"
    "ss, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") or {}\n    confidence = props.get(\"co"
    "nfidence\")\n    socials = props.get(\"socials\") or []\n    websites = props.get(\"websites\") or []\n    instag"
    "ram = first([s for s in (socials + websites)\n                       if isinstance(s, str) and\n           "
    "            re.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n        \"id\": identi"
    "fier[:100],\n        \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],\n        \"categ"
    "ory\": str(taxonomy.get(\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        \"city\": st"
    "r(mun[\"NM_MUN\"])[:120],\n        \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": str(mun"
    "[\"SIGLA_UF\"]),\n        \"address\": str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": l"
    "at, \"longitude\": lon,\n        \"phone\": (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": "
    "(first(props.get(\"emails\")) or \"\")[:200] or None,\n        \"website\": (first_website(websites) or \"\")[:500"
    "] or None,\n        \"instagram\": instagram[:500] if instagram else None,\n        \"confidence\": confidence "
    "if isinstance(confidence, (int, float)) else None,\n        \"operating_status\": str(props.get(\"operating_s"
    "tatus\") or \"\")[:35] or None,\n        \"data_release\": os.getenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n "
    "   }\n\n\ndef lead_quality(row):\n    \"\"\"Prioritize genuine, contactable establishments within the per-city q"
    "uota.\"\"\"\n    confidence = row.get(\"confidence\")\n    weight = float(confidence) * 5 if isinstance(confiden"
    "ce, (int, float)) else 0\n    return (weight\n            + 12 * bool(row.get(\"phone\"))\n            + 11 * "
    "bool(row.get(\"email\"))\n            + 7 * bool(row.get(\"website\"))\n            + 3 * bool(row.get(\"instagr"
    "am\"))\n            + 2 * bool(row.get(\"category\")))\n\n\ndef main():\n    parser = argparse.ArgumentParser(des"
    "cription=__doc__)\n    parser.add_argument(\"--states\", default=\",\".join(STATE_CODES),\n                    "
    "    help=\"Comma-separated UF codes; default all 27\")\n    parser.add_argument(\"--cities\", default=\"\",\n    "
    "                    help=\"Optional comma-separated cities, requires exactly one UF; useful for a small pi"
    "lot\")\n    parser.add_argument(\"--boundaries\", default=None,\n                        help=\"Optional existi"
    "ng ZIP; defaults to one IBGE ZIP per UF\")\n    parser.add_argument(\"--database-max-mb\", type=int, default="
    "400)\n    parser.add_argument(\"--max-records-per-state\", type=int, default=0,\n                        help"
    "=\"0=no state cap; nonzero limits state rows and marks coverage partial\")\n    parser.add_argument(\"--max-r"
    "ecords-per-city\", type=int, default=20,\n                        help=\"Per-municipality safety cap, defaul"
    "t 20 (0=unlimited)\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = parser.parse_ar"
    "gs()\n    if args.database_max_mb < 30 or args.database_max_mb > 480:\n        parser.error(\"--database-max"
    "-mb must be between 30 and 480 MB\")\n    if args.max_records_per_state < 0:\n        parser.error(\"--max-re"
    "cords-per-state cannot be negative\")\n    if not 1 <= args.max_records_per_city <= 100:\n        parser.err"
    "or(\"--max-records-per-city must be between 1 and 100\")\n    states = [s.strip().upper() for s in args.stat"
    "es.split(\",\")]\n    if not states or len(set(states)) != len(states) or any(s not in STATE_CODES for s in "
    "states):\n        parser.error(\"--states must contain unique Brazilian UF codes\")\n    cities = [normalize("
    "s.strip()) for s in args.cities.split(\",\") if s.strip()]\n    if cities and len(states) != 1:\n        pars"
    "er.error(\"--cities requires exactly one UF in --states\")\n\n    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip"
    "(\"/\")\n    key = os.getenv(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    if not args.dry_run and (not base.startswi"
    "th(\"https://\") or not key):\n        parser.error(\"SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY are required"
    "\")\n\n    limit_bytes = args.database_max_mb * 1024 * 1024\n    for state in states:\n        # Individual st"
    "ate archive keeps network and runner-memory use modest.\n        boundaries = ensure_state_boundaries(stat"
    "e, args.boundaries)\n        municipalities = municipalities_index(boundaries)\n        state_munis = munic"
    "ipalities.loc[municipalities.SIGLA_UF == state].copy()\n        if cities:\n            state_munis = state"
    "_munis.loc[\n                state_munis.NM_MUN.map(lambda name: normalize(str(name)) in cities)\n         "
    "   ].copy()\n            found_cities = {normalize(str(name)) for name in state_munis.NM_MUN}\n            "
    "if set(cities) != found_cities:\n                raise RuntimeError(f\"Municipality not found in {state}: {"
    "sorted(set(cities) - found_cities)}\")\n        if state_munis.empty:\n            raise RuntimeError(f\"IBGE"
    " archive does not cover {state}\")\n        west, south, east, north = state_munis.total_bounds\n        # S"
    "lightly wider rectangle avoids border rounding gaps.\n        bbox = \",\".join(f\"{v:.6f}\" for v in (west - "
    ".005, south - .005, east + .005, north + .005))\n        print(f\"[{state}] Overture bbox {bbox}\", flush=Tr"
    "ue)\n        if args.dry_run:\n            continue\n        storage_before = database_bytes(base, key)\n    "
    "    if storage_before >= limit_bytes:\n            print(f\"STOP: database reached configured {args.databas"
    "e_max_mb} MB safety limit.\")\n            return 2\n        total = 0\n        status(base, key, state, \"run"
    "ning\")\n        try:\n            with tempfile.TemporaryDirectory(prefix=\"overture-br-\") as temp:\n        "
    "        out = Path(temp) / f\"{state}.geojsonseq\"\n                subprocess.run([\"overturemaps\", \"downloa"
    "d\", f\"--bbox={bbox}\",\n                                \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(out)]"
    ",\n                               check=True)\n                # Store only a small, quality-ranked per-cit"
    "y selection in memory.\n                city_records = {}\n                city_capped = False\n            "
    "    with out.open(\"r\", encoding=\"utf-8\") as src:\n                    for line in src:\n                   "
    "     line = line.lstrip(\"\\x1e\").strip()\n                        if not line:\n                            "
    "continue\n                        row = place_record(json.loads(line), state_munis, state)\n               "
    "         if row is None:\n                            continue\n                        # Keep two bounded "
    "quality pools per municipality.\n                        # Otherwise the top 30 are almost always ALL \"wit"
    "h site\".\n                        groups = city_records.setdefault(row[\"city_key\"], {\n                    "
    "        \"with_site\": [], \"without_site\": [],\n                        })\n                        pool = gr"
    "oups[\"with_site\" if row.get(\"website\") else \"without_site\"]\n                        if len(pool) < args.m"
    "ax_records_per_city:\n                            pool.append(row)\n                        else:\n         "
    "                   city_capped = True\n                            worst = min(range(len(pool)),\n         "
    "                               key=lambda idx: lead_quality(pool[idx]))\n                            if le"
    "ad_quality(row) > lead_quality(pool[worst]):\n                                pool[worst] = row\n\n         "
    "       # Interleave municipality samples for nationwide fair coverage.\n                ranked_cities = {\n"
    "                    city: select_diverse_pool(groups[\"with_site\"] + groups[\"without_site\"],\n             "
    "                                 args.max_records_per_city)\n                    for city, groups in city_"
    "records.items()\n                }\n                selected = []\n                for rank in range(args.ma"
    "x_records_per_city):\n                    for city in sorted(ranked_cities):\n                        pool "
    "= ranked_cities[city]\n                        if rank < len(pool):\n                            selected.a"
    "ppend(pool[rank])\n\n                if not selected:\n                    raise RuntimeError(\"Overture retu"
    "rned no usable places for this state\")\n                # A municipality-only pilot never marks the whole "
    "state as imported.\n                partial = city_capped or bool(cities)\n                if args.max_reco"
    "rds_per_state and len(selected) > args.max_records_per_state:\n                    selected = selected[:ar"
    "gs.max_records_per_state]\n                    partial = True\n                for start in range(0, len(se"
    "lected), BATCH_SIZE):\n                    if database_bytes(base, key) >= limit_bytes:\n                  "
    "      partial = True\n                        break\n                    batch = selected[start:start + BAT"
    "CH_SIZE]\n                    api(base, key, \"/rest/v1/overture_places?on_conflict=id\", \"POST\", batch)\n   "
    "                 total += len(batch)\n                storage_after = database_bytes(base, key)\n          "
    "      status(base, key, state, \"partial\" if partial else \"complete\", total)\n                delta = stora"
    "ge_after - storage_before\n                bytes_per_imported_row = delta // total if total else 0\n       "
    "         print(f\"[{state}] {total} imported; \"\n                      f\"database before={storage_before}, "
    "after={storage_after}, \"\n                      f\"delta={delta}, average={bytes_per_imported_row} B/row; \""
    "\n                      f\"{'PARTIAL' if partial else 'COMPLETE'}\", flush=True)\n                if storage_"
    "after >= limit_bytes:\n                    print(\"STOP: database safety threshold; remaining states NOT im"
    "ported.\", flush=True)\n                    return 2\n        except Exception as exc:\n            status(ba"
    "se, key, state, \"error\", total, str(exc)[:200])\n            raise\n    print(\"Ingestion pass ended. Check "
    "overture_import_progress for coverage.\", flush=True)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.ex"
    "it(main())\n"
)
_AUTOMATICO = (
    "#!/usr/bin/env python3\n\"\"\"Execução automática Overture, exclusivamente no Supabase STAGING.\n\n- Seleciona "
    "cidades novas a partir do progresso no Supabase.\n- Não modifica produção nem reimporta cidades já present"
    "es.\n- Respeita bloqueio preventivo de armazenamento e guarda falhas para retomar.\n- Usa a mesma seleção m"
    "unicipal dos notebooks, sem API de localidades do IBGE.\n\"\"\"\nimport argparse\nfrom datetime import datetime"
    ", timedelta, timezone\nimport os\nimport subprocess\nimport sys\nimport time\nfrom urllib.parse import urlenco"
    "de\n\nfrom import_overture_br import (\n    STATE_CODES, api, database_bytes, ensure_state_boundaries,\n    m"
    "unicipalities_index, normalize,\n)\n\nSTAGING = \"https://prlrxjothrslhswpkoza.supabase.co\"\nMIB = 1024 * 1024"
    "\nSTOP_MIB = 88  # margem sob o limite estrito do importador (90 MiB)\nPAGE_SIZE = 1000\nPRIORIDADES = {\n   "
    " \"MG\": [\"Uberlândia\", \"Juiz de Fora\", \"Uberaba\", \"Montes Claros\", \"Divinópolis\", \"Contagem\", \"Betim\", \"Se"
    "te Lagoas\"],\n    \"SP\": [\"Sorocaba\", \"São José dos Campos\", \"Santos\", \"Bauru\", \"Piracicaba\", \"São José do "
    "Rio Preto\", \"São Carlos\", \"Franca\"],\n    \"RJ\": [\"Volta Redonda\", \"São Gonçalo\", \"Nova Iguaçu\", \"Duque de "
    "Caxias\", \"Campos dos Goytacazes\", \"Macaé\", \"Cabo Frio\"],\n    \"AC\": [\"Sena Madureira\", \"Brasiléia\", \"Tarau"
    "acá\", \"Feijó\"],\n    \"AL\": [\"Palmeira dos Índios\", \"Rio Largo\", \"Penedo\", \"União dos Palmares\"],\n    \"AP\":"
    " [\"Laranjal do Jari\", \"Oiapoque\", \"Porto Grande\"],\n    \"AM\": [\"Itacoatiara\", \"Manacapuru\", \"Tefé\", \"Coari"
    "\", \"Tabatinga\"],\n    \"BA\": [\"Vitória da Conquista\", \"Camaçari\", \"Ilhéus\", \"Itabuna\", \"Juazeiro\", \"Barreir"
    "as\", \"Porto Seguro\"],\n    \"CE\": [\"Sobral\", \"Maracanaú\", \"Caucaia\", \"Crato\", \"Iguatu\"],\n    \"ES\": [\"Serra\""
    ", \"Cariacica\", \"Cachoeiro de Itapemirim\", \"Linhares\", \"Colatina\"],\n    \"GO\": [\"Anápolis\", \"Rio Verde\", \"T"
    "rindade\", \"Jataí\", \"Itumbiara\", \"Catalão\"],\n    \"MA\": [\"Caxias\", \"Timon\", \"Açailândia\", \"Bacabal\", \"Balsa"
    "s\"],\n    \"MT\": [\"Rondonópolis\", \"Sinop\", \"Sorriso\", \"Tangará da Serra\", \"Cáceres\"],\n    \"MS\": [\"Três Lago"
    "as\", \"Corumbá\", \"Ponta Porã\", \"Naviraí\", \"Nova Andradina\"],\n    \"PA\": [\"Santarém\", \"Marabá\", \"Castanhal\","
    " \"Parauapebas\", \"Altamira\"],\n    \"PB\": [\"Patos\", \"Sousa\", \"Bayeux\", \"Santa Rita\", \"Cajazeiras\"],\n    \"PR\""
    ": [\"Maringá\", \"Cascavel\", \"Ponta Grossa\", \"Foz do Iguaçu\", \"São José dos Pinhais\", \"Guarapuava\"],\n    \"PE"
    "\": [\"Olinda\", \"Paulista\", \"Caruaru\", \"Petrolina\", \"Garanhuns\"],\n    \"PI\": [\"Picos\", \"Floriano\", \"Piripiri"
    "\", \"Campo Maior\", \"Oeiras\"],\n    \"RN\": [\"Parnamirim\", \"Caicó\", \"Macaíba\", \"Assú\", \"Ceará-Mirim\"],\n    \"RS"
    "\": [\"Pelotas\", \"Santa Maria\", \"Canoas\", \"Novo Hamburgo\", \"Passo Fundo\", \"Bento Gonçalves\"],\n    \"RO\": [\"A"
    "riquemes\", \"Vilhena\", \"Cacoal\", \"Jaru\", \"Rolim de Moura\"],\n    \"RR\": [\"Alto Alegre\", \"Caracaraí\", \"Bonfim"
    "\", \"Mucajaí\", \"Cantá\"],\n    \"SC\": [\"Blumenau\", \"Itajaí\", \"Chapecó\", \"Criciúma\", \"São José\", \"Balneário Ca"
    "mboriú\"],\n    \"SE\": [\"Itabaiana\", \"Lagarto\", \"Estância\", \"São Cristóvão\", \"Propriá\"],\n    \"TO\": [\"Gurupi\""
    ", \"Porto Nacional\", \"Paraíso do Tocantins\", \"Colinas do Tocantins\"],\n}\n\nORDEM_UFS = (\"MG SP RJ AC AL AP A"
    "M BA CE ES GO MA MT MS PA PB PR PE PI RN RS RO RR SC SE TO\").split()\nassert set(ORDEM_UFS) == set(STATE_C"
    "ODES) - {\"DF\"}\nassert set(PRIORIDADES) == set(ORDEM_UFS)\n\n\ndef listar(chave, tabela, colunas, filtros=\"\")"
    ":\n    offset = 0\n    tudo = []\n    while True:\n        path = (\"/rest/v1/\" + tabela + \"?select=\" + coluna"
    "s + filtros\n                + \"&limit=\" + str(PAGE_SIZE) + \"&offset=\" + str(offset))\n        pagina = api"
    "(STAGING, chave, path)\n        if not isinstance(pagina, list):\n            raise RuntimeError(\"Falha ao "
    "consultar a tabela \" + tabela)\n        tudo.extend(pagina)\n        if len(pagina) < PAGE_SIZE:\n          "
    "  return tudo\n        offset += len(pagina)\n\n\ndef municipios_existentes(chave):\n    return {\n        (r[\""
    "state\"], r[\"city_key\"])\n        for r in listar(chave, \"overture_import_city_progress\", \"state,city_key\")"
    "\n        if r.get(\"state\") and r.get(\"city_key\")\n    }\n\n\ndef municipios_em_espera(chave):\n    momento = d"
    "atetime.now(timezone.utc).isoformat()\n    filtro = \"&\" + urlencode({\"retry_after\": \"gt.\" + momento})\n    "
    "return {\n        (r[\"state\"], r[\"city_key\"])\n        for r in listar(chave, \"overture_auto_city_attempts\""
    ", \"state,city_key\", filtro)\n        if r.get(\"state\") and r.get(\"city_key\")\n    }\n\n\ndef guardar_falha(cha"
    "ve, uf, cidade, motivo):\n    agora = datetime.now(timezone.utc)\n    proxima = agora + timedelta(days=3)\n "
    "   linha = {\n        \"state\": uf, \"city\": cidade, \"city_key\": normalize(cidade),\n        \"last_error\": st"
    "r(motivo)[:150],\n        \"retry_after\": proxima.isoformat(),\n        \"updated_at\": agora.isoformat(),\n   "
    " }\n    api(STAGING, chave,\n        \"/rest/v1/overture_auto_city_attempts?on_conflict=state,city_key\",\n   "
    "     method=\"POST\", payload=[linha])\n\n\ndef catalogo_malha(uf, cache):\n    if uf not in cache:\n        zip"
    "_path = ensure_state_boundaries(uf)\n        frame = municipalities_index(zip_path)\n        cache[uf] = {\n"
    "            normalize(str(row.NM_MUN)): str(row.NM_MUN)\n            for row in frame.itertuples()\n       "
    "     if str(row.SIGLA_UF) == uf and str(row.NM_MUN).strip()\n        }\n    return cache[uf]\n\n\ndef escolher"
    "(existentes, em_espera, tentadas, cache):\n    contagem = {uf: 0 for uf in ORDEM_UFS}\n    for uf, _ in exi"
    "stentes:\n        if uf in contagem:\n            contagem[uf] += 1\n    ordem = sorted(ORDEM_UFS, key=lambd"
    "a uf: (contagem[uf], ORDEM_UFS.index(uf)))\n    bloqueadas = existentes | em_espera | tentadas\n\n    # Prim"
    "eira passagem: prioridades locais, sem requisição HTTP ao IBGE.\n    for uf in ordem:\n        for cidade i"
    "n PRIORIDADES[uf]:\n            if (uf, normalize(cidade)) not in bloqueadas:\n                return uf, c"
    "idade\n\n    # Quando as prioridades acabam, recorre aos ZIPs oficiais de municípios.\n    for uf in ordem:\n"
    "        for chave, nome in sorted(catalogo_malha(uf, cache).items()):\n            if (uf, chave) not in b"
    "loqueadas:\n                return uf, nome\n    return None\n\n\ndef contagem_cidade(chave, uf, cidade):\n    "
    "filtros = \"&\" + urlencode({\"state\": \"eq.\" + uf, \"city_key\": \"eq.\" + normalize(cidade)})\n    rows = listar"
    "(chave, \"overture_import_city_progress\", \"imported_rows\", filtros)\n    return int(rows[0][\"imported_rows\""
    "]) if rows else 0\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_"
    "argument(\"--cities-per-run\", type=int, default=100)\n    parser.add_argument(\"--max-runtime-minutes\", type"
    "=int, default=40,\n                        help=\"Tempo máximo por sessão, sem depender do GitHub Actions.\""
    ")\n    parser.add_argument(\"--per-city\", type=int, default=60)\n    args = parser.parse_args()\n    if not 1"
    " <= args.cities_per_run <= 300 or not 1 <= args.per_city <= 100:\n        parser.error(\"Use 1..300 municip"
    "ios por sessão e 1..100 empresas por municipio.\")\n    if not 5 <= args.max_runtime_minutes <= 600:\n      "
    "  parser.error(\"--max-runtime-minutes deve estar entre 5 e 600 minutos.\")\n\n    # Evita que uma variável d"
    "e ambiente ou secret de produção seja usada.\n    if os.environ.get(\"SUPABASE_URL\", \"\").rstrip(\"/\") != STA"
    "GING:\n        raise SystemExit(\"BLOQUEADO: SUPABASE_URL nao corresponde ao STAGING autorizado.\")\n    segr"
    "edo = os.environ.get(\"SUPABASE_SERVICE_ROLE_KEY\", \"\").strip()\n    if not segredo.startswith((\"sb_secret_\""
    ", \"eyJ\")):\n        raise SystemExit(\"BLOQUEADO: segredo de homologacao nao configurado/valido.\")\n\n    dea"
    "dline = time.monotonic() + args.max_runtime_minutes * 60\n    antes = database_bytes(STAGING, segredo)\n   "
    " print(f\"STAGING: {antes/MIB:.2f} MiB. Limite preventivo: {STOP_MIB} MiB.\", flush=True)\n    print(f\"Modo "
    "contínuo: até {args.max_runtime_minutes} minutos, \"\n          f\"máximo de {args.cities_per_run} município"
    "s por sessão.\", flush=True)\n    if antes >= STOP_MIB * MIB:\n        print(\"PARADA SEGURA: armazenamento a"
    "cima do limite preventivo.\")\n        return 0\n\n    existentes = municipios_existentes(segredo)\n    em_esp"
    "era = municipios_em_espera(segredo)\n    print(f\"Municipios existentes: {len(existentes)}; em espera: {len"
    "(em_espera)}\")\n    tentadas = set()\n    catalogos = {}\n    importadas = 0\n    falhas = 0\n    while import"
    "adas < args.cities_per_run and falhas < 3:\n        remaining = deadline - time.monotonic()\n        if rem"
    "aining < 240:\n            print(\"PAUSA PARA RETOMADA: sessão próxima do limite de tempo.\", flush=True)\n  "
    "          break\n        if database_bytes(STAGING, segredo) >= STOP_MIB * MIB:\n            print(\"PARADA "
    "SEGURA: banco atingiu limite preventivo.\")\n            break\n        proxima = escolher(existentes, em_es"
    "pera, tentadas, catalogos)\n        if not proxima:\n            print(\"SEM PROXIMOS MUNICIPIOS ELEGIVEIS.\""
    ")\n            break\n        uf, cidade = proxima\n        item = (uf, normalize(cidade))\n        tentadas."
    "add(item)\n        if contagem_cidade(segredo, uf, cidade) > 0:\n            existentes.add(item)\n         "
    "   continue\n        print(f\"INICIANDO: {cidade}/{uf} (ate {args.per_city})\", flush=True)\n\n        cmd = ["
    "\n            sys.executable, \"scripts/import_overture_br.py\",\n            \"--states\", uf, \"--cities\", cid"
    "ade,\n            \"--max-records-per-state\", str(args.per_city),\n            \"--max-records-per-city\", str"
    "(args.per_city),\n            \"--database-max-mb\", \"90\",\n        ]\n        try:\n            # Máximo 12 mi"
    "nutos por município; sempre deixa margem para\n            # salvar o resultado e encerrar a sessão antes "
    "do timeout do GitHub.\n            import_timeout = min(720, max(60, int(deadline - time.monotonic() - 120"
    ")))\n            proc = subprocess.run(cmd, timeout=import_timeout, check=False)\n            if proc.retur"
    "ncode != 0:\n                # Retentativa em 3 dias para não ficar preso na mesma cidade.\n               "
    " guardar_falha(segredo, uf, cidade, \"exit_code=\" + str(proc.returncode))\n                em_espera.add(it"
    "em)\n                falhas += 1\n                print(f\"ADIADO: {cidade}/{uf}; retorno {proc.returncode}."
    "\")\n                continue\n        except subprocess.TimeoutExpired:\n            if deadline - time.mono"
    "tonic() < 180:\n                print(f\"PAUSA PARA RETOMADA: limite de tempo em {cidade}/{uf}. \"\n         "
    "             \"O banco preserva os registros já gravados.\", flush=True)\n                break\n            "
    "guardar_falha(segredo, uf, cidade, \"timeout_ate_720s\")\n            em_espera.add(item)\n            falhas"
    " += 1\n            print(f\"ADIADO: {cidade}/{uf}; tempo excedido.\", flush=True)\n            continue\n\n    "
    "    depois = contagem_cidade(segredo, uf, cidade)\n        if depois == 0:\n            guardar_falha(segre"
    "do, uf, cidade, \"sem_registros_verificados\")\n            em_espera.add(item)\n            falhas += 1\n    "
    "        print(f\"ADIADO: {cidade}/{uf}; nenhum registro confirmado.\")\n            continue\n        existen"
    "tes.add(item)\n        importadas += 1\n        print(f\"CONFIRMADO NO SUPABASE: {cidade}/{uf} | {depois} em"
    "presas\", flush=True)\n\n    print(f\"SESSAO CONCLUIDA: {importadas} novos municípios; {falhas} falhas adiada"
    "s; \"\n          f\"{len(existentes)} municípios no banco; \"\n          f\"{database_bytes(STAGING, segredo)/M"
    "IB:.2f} MiB. \"\n          \"Quando habilitado, o agendamento fará a próxima sessão.\", flush=True)\n    retur"
    "n 0 if falhas == 0 else 1\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
)
scripts = Path("/kaggle/working/scripts")
scripts.mkdir(parents=True, exist_ok=True)
for file_name, content in (
    ("import_overture_br.py", _IMPORTADOR),
    ("import_overture_staging_auto.py", _AUTOMATICO),
):
    file_path = scripts / file_name
    compile(content, str(file_path), "exec")
    file_path.write_text(content, encoding="utf-8")
print("Importadores preparados: STAGING obrigatório, limite preventivo 88 MiB.")


In [ ]:
# IMPORTAÇÃO REAL. O segredo fica SOMENTE em Kaggle Add-ons > Secrets.
# Nunca escreva sua chave dentro do notebook nem compartilhe seus logs privados.
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os
import subprocess
import sys

SEGREDO_NOME = "OVERTURE_SUPABASE_SERVICE_ROLE_KEY"
try:
    segredo = UserSecretsClient().get_secret(SEGREDO_NOME)
except Exception:
    raise RuntimeError(
        "Segredo não disponível. Crie OVERTURE_SUPABASE_SERVICE_ROLE_KEY "
        "em Add-ons > Secrets e conceda acesso a este notebook."
    ) from None
if not segredo or not segredo.strip().startswith(("sb_secret_", "eyJ")):
    raise RuntimeError("Use a chave SECRETA do Supabase STAGING; nunca a pública.")
# A URL é imutável, não recebe substituição pelo usuário.
ambiente = os.environ.copy()
ambiente["SUPABASE_URL"] = "https://prlrxjothrslhswpkoza.supabase.co"
ambiente["SUPABASE_SERVICE_ROLE_KEY"] = segredo.strip()
segredo = None

comando = [
    sys.executable, "-u", "/kaggle/working/scripts/import_overture_staging_auto.py",
    "--cities-per-run", str(MAX_MUNICIPIOS),
    "--per-city", str(MAX_EMPRESAS),
    "--max-runtime-minutes", str(MAX_MINUTOS),
]
print("INICIANDO: importação na homologação, até", MAX_MUNICIPIOS,
      "municípios. Chave oculta.", flush=True)
try:
    # A sessão Kaggle pode encerrar antes por limites da plataforma.
    subprocess.run(
        comando,
        env=ambiente, check=True, timeout=MAX_MINUTOS * 60 + 300,
        cwd="/kaggle/working",
    )
    print("Sessão concluída. Municípios registrados serão pulados na próxima execução.")
except subprocess.TimeoutExpired:
    print("A execução ultrapassou o tempo reservado. Confira o estado no Supabase.")
    raise
finally:
    ambiente.pop("SUPABASE_SERVICE_ROLE_KEY", None)
    del ambiente
